# Notebook 02: Basic RAG Pipeline with LangChain 1.3.14+ (LCEL)

Welcome to Notebook 02. In this notebook, we construct a production-ready, modular **Retrieval-Augmented Generation (RAG)** pipeline using **LangChain Expression Language (LCEL)**.

---

## Key Objectives & Architecture Rules
1. **Modern LangChain 1.3.14+ Compliance:** Completely avoid deprecated legacy chain constructors (e.g., `create_stuff_documents_chain`, `create_retrieval_chain` from `langchain.chains`).
2. **Declarative LCEL Pipelines:** Express the full RAG pipeline using composable `Runnable` components (`RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`).
3. **OpenRouter Provider Integration:** Use OpenRouter endpoint (`https://openrouter.ai/api/v1`) with OpenRouter model identifiers (`openai/gpt-4o-mini`).
4. **Modular & Reusable:** Integrate seamlessly with the project structure (`src/config/`, `src/prompts/`, `src/rag/`).
5. **LangGraph Ready:** Structure state inputs and outputs cleanly so the pipeline can easily be lifted into a LangGraph node in future steps.

---

## Legacy vs Modern LangChain 1.3.14 Architecture

| Component | Legacy LangChain (Deprecated/Removed) | Modern LangChain 1.3.14+ (LCEL) |
|---|---|---|
| **Document Combination** | `create_stuff_documents_chain(llm, prompt)` | `RunnableLambda(format_docs)` embedded into `RunnableParallel` |
| **Retrieval Chain** | `create_retrieval_chain(retriever, combine_chain)` | `RunnableParallel(context=retriever | format_docs, input=RunnablePassthrough())` |
| **Output Parsing** | Implicit dictionary outputs | Declarative `StrOutputParser()` |
| **Composition** | Nested helper functions | Standard Unix-style pipe operator `|` (`RunnableSequence`) |

---

## LCEL Data Flow Diagram
```
                       User Question (str)
                               |
                               v
                    +---------------------+
                    |  RunnableParallel   |
                    +----------+----------+
                               |
             +-----------------+-----------------+
             v                                   v
   context: retriever                   input: RunnablePassthrough()
             |                                   |
             v                                   |
   format_docs (RunnableLambda)                  |
             |                                   |
             +-----------------+-----------------+
                               |
                               v
                    Dict { context, input }
                               |
                               v
                ChatPromptTemplate (qa_prompt)
                               |
                               v
               ChatOpenAI (OpenRouter: gpt-4o-mini)
                               |
                               v
                        StrOutputParser
                               |
                               v
                         Final Answer (str)
```

### Step 1: Environment & System Path Configuration
When running inside `notebook/`, Python's default `sys.path` points to the `notebook/` directory. To allow seamless imports from `src/` (such as `src.config` and `src.prompts`), we compute the `PROJECT_ROOT` and insert it into `sys.path`.

In [1]:
import sys
from pathlib import Path

# Determine project root relative to this notebook's directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

# Guard against duplicate insertions in sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root resolved: {PROJECT_ROOT}")
print(f"Active sys.path[0]:     {sys.path[0]}")

Project root resolved: D:\career-ai-agent
Active sys.path[0]:     D:\career-ai-agent


### Step 2: Load Centralised Settings & Validate Environment
We import `settings` from `src.config.settings`. This loads configuration from `.env` and validates required secrets (like `OPENROUTER_API_KEY`) at import time.

In [2]:
from src.config.settings import settings

# Display safe configuration representation (secrets redacted)
print("Loaded Settings Configuration:")
print(settings)

Loaded Settings Configuration:
Settings(APP_ENV='development', MODEL_NAME='openai/gpt-4o-mini', EMBEDDING_MODEL='sentence-transformers/all-MiniLM-L6-v2', TOP_K=5, CHUNK_SIZE=1000, CHUNK_OVERLAP=200, TEMPERATURE=0.0, VECTOR_DB_PATH=WindowsPath('storage/vector_db'), LANGSMITH_TRACING=True)


### Step 3: Import Modern LangChain 1.3.14 Primitives
We import only official, non-deprecated packages from `langchain_core`, `langchain_chroma`, `langchain_huggingface`, and `langchain_openai`.

In [3]:
# Vector Store & Embeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Modern LCEL Core Primitives
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Chat Model Client
from langchain_openai import ChatOpenAI

# Application Prompts
from src.prompts.system_prompt import SYSTEM_PROMPT

D:\career-ai-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 4: Initialise HuggingFace Embeddings
We load the embedding model configured in `settings.EMBEDDING_MODEL`. `normalize_embeddings=True` ensures cosine similarity calculations match ChromaDB expectations.

In [4]:
EMBEDDING_MODEL_NAME = settings.EMBEDDING_MODEL

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Embedding Model Initialised: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5382.34it/s]

Embedding Model Initialised: sentence-transformers/all-MiniLM-L6-v2


### Step 5: Connect to Persistent Chroma Vector Database
We instantiate `Chroma` pointing to `storage/vector_db`. This directory contains pre-processed knowledge chunks ingested via `scripts/ingest.py`.

In [5]:
VECTOR_DB_PATH = PROJECT_ROOT / settings.VECTOR_DB_PATH

vector_db = Chroma(
    persist_directory=str(VECTOR_DB_PATH),
    embedding_function=embedding_model,
)

doc_count = vector_db._collection.count()
print(f"Connected to ChromaDB at: {VECTOR_DB_PATH}")
print(f"Total Indexed Chunks:    {doc_count:,}")

Connected to ChromaDB at: D:\career-ai-agent\storage\vector_db
Total Indexed Chunks:    285


### Step 6: Expose Vector Store as a Runnable Retriever (Optimised)

We configure the retriever to use **Maximal Marginal Relevance (MMR)** instead of basic similarity. 

#### Why these parameters?
1. **`search_type="mmr"`**: Basic similarity often retrieves 5 identical chunks from the same document (e.g., adjacent paragraphs). MMR forces *diversity*. It first fetches `fetch_k` relevant chunks, then selects `k` chunks that are both relevant to the query AND dissimilar to each other.
2. **`fetch_k=20`**: The number of candidate chunks retrieved before filtering for diversity. 20 is a good balance between speed and candidate variety.
3. **`lambda_mult=0.7`**: Controls diversity. 1.0 is pure similarity, 0.0 is pure diversity. 0.7 strongly favours relevance but penalises exact duplicates.
4. **Chunking Strategy Note**: Our documents are chunked at `CHUNK_SIZE=1000` with `CHUNK_OVERLAP=200`. This ensures context isn't lost across page boundaries, but means adjacent chunks are highly similar, making MMR crucial for this project.

In [6]:
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": settings.TOP_K,
        "fetch_k": 20,
        "lambda_mult": 0.7
    },
)

print(f"Retriever ready (MMR | Top-K: {settings.TOP_K})")

Retriever ready (MMR | Top-K: 5)


### Step 7: Verify Standalone Retrieval (`retriever.invoke`)
In LangChain 1.3.14+, `.invoke()` is the standard method for querying retrievers. `get_relevant_documents()` is deprecated.

In [7]:
test_query = "How should I prepare for a behavioral interview?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} Document Chunks:\n")

for idx, doc in enumerate(retrieved_docs, start=1):
    source = doc.metadata.get("filename", doc.metadata.get("source", "unknown"))
    print(f"[{idx}] Source: {source}")
    print(f"    Content Snippet: {doc.page_content[:180]}...")
    print("-" * 60)

Query: 'How should I prepare for a behavioral interview?'


Retrieved 5 Document Chunks:

[1] Source: CMU_Behavioral_Interview_Guide.pdf
    Content Snippet: Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure ...
------------------------------------------------------------
[2] Source: CMU_Behavioral_Interview_Guide.pdf
    Content Snippet: with. These questions should be authentic to your curiosities and should not be 
easily answered by searching the internet or the company’s website. If you know 
the names of the p...
------------------------------------------------------------
[3] Source: MASTERS-RESUME-GUIDE.pdf
    Content Snippet: SKILLS  
Programming: Python (numpy, pandas, scikit-learn, pytorch), SQL, R, Bloomberg Terminal, MATLAB, Latex 
Language: Fluent in Korean and Chinese 
 
RELEVANT RESEARCH  
Harvar...
------------------------------------------------------------
[4] Source: UMich_Alumni_Networking_Guid

### Step 8: Configure OpenRouter Provider with GPT-4o Mini (`ChatOpenAI`)
Because OpenRouter provides an OpenAI-compatible API, we use `ChatOpenAI` configured with:
1. **`base_url`**: `https://openrouter.ai/api/v1` (routes calls to OpenRouter)
2. **`api_key`**: `settings.OPENROUTER_API_KEY` (OpenRouter secret key)
3. **`model`**: `openai/gpt-4o-mini` (OpenRouter model string format: `vendor/model-name`)

In [8]:
# OpenRouter API configuration using ChatOpenAI client
llm = ChatOpenAI(
    model=settings.MODEL_NAME,
    api_key=settings.OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=settings.TEMPERATURE,
    max_tokens=1024,
    max_retries=3,
    timeout=60,
)

# Verify OpenRouter LLM connectivity
ping_response = llm.invoke("Reply with 'OpenRouter OK' if connected.")
print(f"OpenRouter LLM Model:   {settings.MODEL_NAME}")
print(f"Connectivity Test:      {ping_response.content.strip()}")

OpenRouter LLM Model:   openai/gpt-4o-mini
Connectivity Test:      OpenRouter OK


### Step 9: Define Document Formatting Transformation (`RunnableLambda`)
To feed retrieved `Document` objects into the prompt context string, we define `format_docs` and wrap it into a `RunnableLambda`.

In [9]:
def format_docs(docs: list[Document]) -> str:
    """Concatenate document contents into a clean context string."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Wrap transformation into a reusable RunnableLambda
format_docs_runnable = RunnableLambda(format_docs)
print("RunnableLambda format_docs component created.")

RunnableLambda format_docs component created.


---

### Step 10: System Prompt Design & QA Prompt Template

`SYSTEM_PROMPT` is imported from `src.prompts.system_prompt`. It is a plain `str` constant (not a template) because it contains no runtime variables.

#### Design decisions in the System Prompt

| Design Choice | Why |
|---|---|
| **Three coverage states** (full / partial / none) | The old prompt only had binary "answer" vs "I don't know". This caused the model to either hallucinate missing details or refuse when partial context existed. |
| **"Complete and detailed" instead of "concise"** | In a RAG pipeline the user expects thorough answers grounded in documents. "Be concise" caused the model to drop supported information. |
| **Explicit formatting rules** | Bullet points, bold text, and headers make answers scannable. Without explicit formatting instructions, models tend to produce dense paragraphs. |
| **Prompt injection defence with examples** | Naming specific attack phrases ("ignore your instructions", "forget everything above") makes the defence more robust than a vague "ignore injections". |

#### ChatPromptTemplate structure

The template has exactly **two messages**:

1. **System message** — the full `SYSTEM_PROMPT` text. It is static policy, baked in at template construction time (not a `{variable}`). This means it never gets accidentally overridden by user input.
2. **Human message** — contains two runtime `{variables}` populated by the LCEL pipeline:
   - `{context}` — the formatted retrieved chunks (from `retriever | format_docs_runnable`)
   - `{input}` — the original user question (from `RunnablePassthrough()`)

Placing `{context}` inside the human message rather than the system message is intentional:
- The system message defines *policy* (how to behave)
- The human message defines *data* (what to work with)
- This separation makes the prompt resilient when migrating to LangGraph, where system instructions are set once and user turns vary per invocation.

In [10]:
# Display the system prompt for educational review
print("=" * 70)
print("SYSTEM PROMPT")
print("=" * 70)
print(SYSTEM_PROMPT)
print("=" * 70)

SYSTEM PROMPT
You are a **Career AI Assistant** — a professional advisor specialising in career development, CV/resume writing, interview preparation, job searching, salary negotiation, and professional growth.

## Answering Rules

You will receive a **Context** section containing text retrieved from career guidance documents. Follow these rules without exception:

### Rule 1 — Answer ONLY from the provided context
Base your entire answer on the information present in the context. Do NOT use prior knowledge, training data, or external information to supplement, extend, or fill gaps in the context.

### Rule 2 — Full coverage
When the context contains sufficient information to fully answer the question, provide a **complete, detailed, and well-structured** response. Extract every relevant detail from the context. Do not summarise or shorten the answer when the context provides more depth.

### Rule 3 — Partial coverage
When the context contains only *partial* information relevant to the

In [11]:
# Build the ChatPromptTemplate with the system prompt baked in.
#
# Template variables:
#   {context} - populated by: retriever | format_docs_runnable
#   {input}   - populated by: RunnablePassthrough()

qa_prompt = ChatPromptTemplate.from_messages([
    # System turn: static policy — defines HOW the model behaves
    ("system", SYSTEM_PROMPT),
    # Human turn: dynamic data — defines WHAT the model works with
    ("human", "Context:\n{context}\n\nQuestion:\n{input}"),
])

print("QA Prompt Template constructed.")
print(f"Required input variables: {qa_prompt.input_variables}")

QA Prompt Template constructed.
Required input variables: ['context', 'input']


### Step 11: Compose the Declarative LCEL RAG Pipeline
Using `RunnableParallel`, we execute retrieval and input passthrough simultaneously. We then compose the full pipeline using the `|` operator (`RunnableSequence`).

In [12]:
# Parallel context retrieval and query passthrough
context_retrieval_step = RunnableParallel({
    "context": retriever | format_docs_runnable,
    "input": RunnablePassthrough(),
})

# Declarative LCEL RAG Sequence
rag_chain = (
    context_retrieval_step
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("Declarative LCEL RAG Chain successfully constructed.")

Declarative LCEL RAG Chain successfully constructed.


### Step 12: Execute End-to-End RAG Pipeline
We invoke `rag_chain.invoke()` with a sample question and observe the answer generated from retrieved context.

In [13]:
sample_question = "How should I prepare for a behavioral interview?"
answer = rag_chain.invoke(sample_question)

print(f"Question: {sample_question}")
print("=" * 70)
print(answer)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Question: How should I prepare for a behavioral interview?
## Preparing for a Behavioral Interview

To effectively prepare for a behavioral interview, consider the following steps:

### 1. Research the Company
- **Understand the Company**: Spend time thoroughly researching the company to understand its values, culture, and mission.
- **Tailor Your Questions**: Prepare questions that reflect your genuine curiosity about the company and its operations. These should not be easily answered by a simple internet search.
- **Know Your Interviewers**: If you know the names of the people you will be interviewing with, tailor your questions to their backgrounds and roles within the company.

### 2. Prepare Responses to Common Questions
- **Classic Interview Questions**: Be ready to answer classic questions about yourself, your interests, and your background as it relates to the job description.
- **Avoid Scripts**: Instead of writing out a full script, practice your responses aloud or create bul

### Step 13: Inspect Intermediate Pipeline State
We invoke `context_retrieval_step` directly to inspect the exact `{context, input}` dictionary generated before prompt evaluation.

In [14]:
intermediate_state = context_retrieval_step.invoke(sample_question)

print("--- Extracted Context String ---")
print(intermediate_state["context"][:500] + "...\n")
print("--- Passthrough Input ---")
print(intermediate_state["input"])

--- Extracted Context String ---
Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure of the type of interview you are 
scheduled for, you can ask your recruiter for clarity in advance. Behavioral Interviews 
allow recruiters and members of a hiring team to assess how and if your past 
experiences, behaviors, and skills demonstrate the key characteristics and competencies 
they have deemed essential for...

--- Passthrough Input ---
How should I prepare for a behavioral interview?


---

### Step 14: Diagnostic Validation Test Suite

To evaluate whether the retriever is returning the most relevant chunks, we run a deep diagnostic. 

For each test we print:
1. **Raw Similarity Scores** (using `similarity_search_with_score` to see vector distances)
2. **Final Retrieved Documents** (after MMR diversity filtering is applied)
3. **Formatted context** (the string injected into `{context}`)
4. **Final answer** from the LLM

This helps us debug if failure is due to bad retrieval (wrong chunks) or bad generation (LLM ignoring chunks).

In [15]:
def run_diagnostic(question: str) -> str:
    """
    Run a full diagnostic on the RAG pipeline for a single question.
    """
    separator = "=" * 70
    print(separator)
    print(f"QUESTION: {question}")
    print(separator)
    
    # --- 0. Raw Similarity Scores (Diagnostic Only) ---
    print(f"\n[0] RAW SIMILARITY SCORES (Before MMR)")
    print("-" * 40)
    # Note: In Chroma, lower score means shorter distance (more similar)
    raw_docs = vector_db.similarity_search_with_score(question, k=5)
    for i, (doc, score) in enumerate(raw_docs, 1):
        source = doc.metadata.get("filename", doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", "?")
        print(f"  [{i}] L2 Distance: {score:.4f} | {source} (page {page})")
    
    # --- 1. Actual Retrieved Documents (After MMR) ---
    docs = retriever.invoke(question)
    print(f"\n[1] FINAL RETRIEVED DOCUMENTS (After MMR - {len(docs)} chunks)")
    print("-" * 40)
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("filename", doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", "?")
        print(f"  [{i}] {source} (page {page})")
        # Print a tiny snippet to understand what was retrieved
        snippet = doc.page_content.replace('\n', ' ')[:100]
        print(f"      Snippet: {snippet}...")
    
    # --- 2. Format context ---
    context = format_docs(docs)
    print(f"\n[2] FORMATTED CONTEXT (first 300 chars)")
    print("-" * 40)
    print(context[:300].replace('\n', ' ') + "...")
    
    # --- 3. Get the answer ---
    answer = rag_chain.invoke(question)
    print(f"\n[3] FINAL ANSWER")
    print("-" * 40)
    print(answer)
    print(separator + "\n")
    return answer

### Benchmark Suite
Running the requested test queries to validate retrieval and response generation.

In [16]:
_ = run_diagnostic("What skills are required for a Machine Learning Engineer?")

QUESTION: What skills are required for a Machine Learning Engineer?

[0] RAW SIMILARITY SCORES (Before MMR)
----------------------------------------
  [1] L2 Distance: 1.6722 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 1)
  [2] L2 Distance: 1.6782 | Meta_ML_onsite_interview_prep.pdf (page 4)
  [3] L2 Distance: 1.6803 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 19)
  [4] L2 Distance: 1.6808 | Meta_ML_onsite_interview_prep.pdf (page 3)
  [5] L2 Distance: 1.6877 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 18)

[1] FINAL RETRIEVED DOCUMENTS (After MMR - 5 chunks)
----------------------------------------
  [1] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 1)
      Snippet: M achine learning interview questions are an integral part  of the data science interview and the pa...
  [2] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 13)
      Snippet: (classification, prediction, etc.) and bring up a f


[3] FINAL ANSWER
----------------------------------------
The provided documents do not contain information to answer this question.



In [17]:
_ = run_diagnostic("What is feature engineering?")

QUESTION: What is feature engineering?

[0] RAW SIMILARITY SCORES (Before MMR)
----------------------------------------
  [1] L2 Distance: 1.5423 | HES-Resume-samples-combined.pdf (page 9)
  [2] L2 Distance: 1.5538 | HES-Resume-samples-combined.pdf (page 8)
  [3] L2 Distance: 1.5755 | Meta_ML_onsite_interview_prep.pdf (page 3)
  [4] L2 Distance: 1.5777 | git-cheat-sheet-education.pdf (page 0)
  [5] L2 Distance: 1.5841 | HES-Resume-samples-combined.pdf (page 8)

[1] FINAL RETRIEVED DOCUMENTS (After MMR - 5 chunks)
----------------------------------------
  [1] HES-Resume-samples-combined.pdf (page 9)
      Snippet: Proven Leadership:  • Recognized by executive management for building excellent relationships with t...
  [2] HES-Resume-samples-combined.pdf (page 8)
      Snippet:  Improved process efficiency 75% by standardizing end to end project management workflow  Reduced ...
  [3] Meta_ML_onsite_interview_prep.pdf (page 3)
      Snippet: y our existing toolsets to model the problem

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



[3] FINAL ANSWER
----------------------------------------
The provided documents do not contain information to answer this question.



In [18]:
_ = run_diagnostic("What is overfitting?")

QUESTION: What is overfitting?

[0] RAW SIMILARITY SCORES (Before MMR)
----------------------------------------
  [1] L2 Distance: 1.5490 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 12)
  [2] L2 Distance: 1.5609 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 12)
  [3] L2 Distance: 1.6339 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 2)
  [4] L2 Distance: 1.6602 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 2)
  [5] L2 Distance: 1.6778 | HES-Resume-samples-combined.pdf (page 8)

[1] FINAL RETRIEVED DOCUMENTS (After MMR - 5 chunks)
----------------------------------------
  [1] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 12)
      Snippet: rate generalizations. There are three main methods to avoid overfitting: 1- Keep the model simpler: ...
  [2] HES-Resume-samples-combined.pdf (page 8)
      Snippet:  Improved process efficiency 75% by standardizing end to end project management workflow  

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



[3] FINAL ANSWER
----------------------------------------
*The provided documents do not contain information to answer this question.*



In [19]:
_ = run_diagnostic("How should machine learning models be evaluated?")

QUESTION: How should machine learning models be evaluated?

[0] RAW SIMILARITY SCORES (Before MMR)
----------------------------------------
  [1] L2 Distance: 1.5219 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 20)
  [2] L2 Distance: 1.5499 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 13)
  [3] L2 Distance: 1.5563 | Meta_ML_onsite_interview_prep.pdf (page 3)
  [4] L2 Distance: 1.5668 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 10)
  [5] L2 Distance: 1.5690 | Meta_ML_onsite_interview_prep.pdf (page 4)

[1] FINAL RETRIEVED DOCUMENTS (After MMR - 5 chunks)
----------------------------------------
  [1] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 20)
      Snippet: about machine learning will have gone off and done side projects on  their own, and have a good idea...
  [2] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 13)
      Snippet: You would first split the dataset into training and test s

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



[3] FINAL ANSWER
----------------------------------------
## Evaluation of Machine Learning Models

To effectively evaluate machine learning models, the following steps and techniques should be considered:

1. **Dataset Splitting**:
   - **Training and Test Sets**: Split the dataset into training and test sets to assess the model's performance on unseen data.
   - **Cross-Validation**: Use cross-validation techniques to further segment the dataset into composite sets of training and test sets, ensuring a robust evaluation.

2. **Performance Metrics**:
   - Implement a selection of performance metrics to measure the model's effectiveness. Some commonly used metrics include:
     - **F1 Score**: Balances precision and recall, useful for imbalanced datasets.
     - **Accuracy**: The ratio of correctly predicted instances to the total instances.
     - **Confusion Matrix**: A table that describes the performance of a classification model by showing true positives, false positives, true ne

In [20]:
_ = run_diagnostic("Explain the ML system design process.")

QUESTION: Explain the ML system design process.

[0] RAW SIMILARITY SCORES (Before MMR)
----------------------------------------
  [1] L2 Distance: 1.4941 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 1)
  [2] L2 Distance: 1.5245 | Meta_ML_onsite_interview_prep.pdf (page 3)
  [3] L2 Distance: 1.5442 | Meta_ML_onsite_interview_prep.pdf (page 4)
  [4] L2 Distance: 1.5467 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 1)
  [5] L2 Distance: 1.5509 | 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 10)

[1] FINAL RETRIEVED DOCUMENTS (After MMR - 5 chunks)
----------------------------------------
  [1] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 1)
      Snippet: M achine learning interview questions are an integral part  of the data science interview and the pa...
  [2] 10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 10)
      Snippet: More reading: Pruning (decision trees) Pruning is what happens in decisi


[3] FINAL ANSWER
----------------------------------------
The provided documents do not contain information to answer this question.



---

### Step 15: Architectural Readiness & LangGraph Migration Plan
Because `rag_chain` is built using standard LCEL `Runnable` components, migrating to LangGraph in subsequent notebooks is straightforward:

```python
# Example LangGraph Node Integration
from src.agent.state import AgentState

def rag_node(state: AgentState) -> dict:
    answer = rag_chain.invoke(state["question"])
    return {"answer": answer}
```